# NVIDIA LABS Hybrid Solver — Run Notebook (CLI-first repo)

This notebook is an **interactive “runbook”** around the CLI-first repo you unzipped.

It does **not** replace the repository’s reproducible scripts; it simply:
- configures `sys.path` so you can import the package,
- runs **Gate 0** tests,
- runs **one end-to-end experiment** (Seeder → optional post-select → fixed MTS),
- optionally checks **CPU↔GPU parity** (if a CUDA GPU + CuPy are available),
- shows where outputs land (`results/…`) and how to package the **handoff bundle**.

> **Non-negotiable rule reminder:** we do **not** inject new ideas inside the MTS loop. All “twists” live in the seeders.


## 0) Point Python at the repo (`src/`) and sanity-check the environment


In [ ]:
import os, sys, platform
from pathlib import Path

# Set REPO_ROOT to the folder that contains `src/`, `scripts/`, `tests/`
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repo root (the folder that contains src/).")

# Make imports work without installing a wheel
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Repo root:", REPO_ROOT)


In [ ]:
# Optional: check whether GPU runtime is visible
try:
    import cupy as cp
    n = int(cp.cuda.runtime.getDeviceCount())
    print("CuPy OK. CUDA devices:", n)
except Exception as e:
    cp = None
    print("CuPy not available or no CUDA runtime:", repr(e))

# Optional: check whether CUDA-Q is importable (used by quantum seeders)
try:
    import cudaq  # type: ignore
    print("CUDA-Q import OK:", getattr(cudaq, "__version__", "unknown"))
except Exception as e:
    print("CUDA-Q import failed (seeders will fall back to skeleton behavior where applicable):", repr(e))


## 1) Gate 0 — correctness tests (CPU oracle + parity checks)

Run this first whenever you change code. If this fails, fix correctness before benchmarking.


In [ ]:
import subprocess, textwrap

# Run pytest from the repo root
cmd = [sys.executable, "-m", "pytest", "-q"]
print("Running:", " ".join(cmd))
p = subprocess.run(cmd, cwd=str(REPO_ROOT))
print("\npytest return code:", p.returncode)


## 2) Run **one end-to-end experiment** from Python (no subprocess)

This mirrors what the CLI does:
1. generate seeds via chosen seeder,
2. (optional) post-select best K by energy,
3. run **fixed** MTS (CPU or GPU),
4. write logs / artifacts in a new run folder.


In [ ]:
from labs_hybrid.config import RunConfig, SeederConfig
from labs_hybrid.logging_utils import new_run_dir, write_json
from labs_hybrid.pipeline import run_experiment

# Choose a small N for a fast demo
N = 16
seeder = "random"  # try: "qaoa", "dcqo", "dcqo_plus", "pce"

run_dir = new_run_dir("results", f"notebook_{seeder}")
cfg = RunConfig(
    N=N,
    seeder=SeederConfig(
        name=seeder,
        shots=256,
        K_out=256,
        K_select=32,
        params={}  # seeder-specific hyperparams go here
    ),
    backend="auto",  # seeder backend request: "cpu" | "gpu" | "auto"
    device="auto",   # MTS device request: "cpu" | "gpu" | "auto"
    budget_s=1.0,
    max_iters=None,
    rng_seed=0,
    out_dir=str(run_dir),
    run_tag="notebook",
    save_traces=True,
)

write_json(Path(run_dir) / "run_config.json", cfg.to_dict())
row = run_experiment(cfg)

print("Wrote to:", run_dir)
row


### Inspect artifacts: best-so-far curve and seed energy histogram


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

artifact_prefix = row["artifact_prefix"]
artifacts = Path(row["out_dir"]) / "artifacts"

trace_path = artifacts / f"{artifact_prefix}_mts_trace.json"
seedE_path = artifacts / f"{artifact_prefix}_seed_energies.npy"

print("Trace:", trace_path.exists(), trace_path)
print("Seed energies:", seedE_path.exists(), seedE_path)

if trace_path.exists():
    trace = json.loads(trace_path.read_text())
    best_trace = trace.get("best_trace", [])
    if best_trace:
        plt.figure()
        plt.plot(best_trace)
        plt.title("Best-so-far energy vs iteration")
        plt.xlabel("iteration")
        plt.ylabel("energy (lower is better)")
        plt.show()

if seedE_path.exists():
    seedE = np.load(seedE_path)
    plt.figure()
    plt.hist(seedE, bins=30)
    plt.title("Seed energy histogram (shows left-tail advantage)")
    plt.xlabel("energy")
    plt.ylabel("count")
    plt.show()


## 3) Run the CLI (same pipeline, reproducible logs)

Use this when you want clean, script-generated outputs (judge-friendly).

Example: one CPU run with DCQO+ and explicit trotter knobs.


In [ ]:
import subprocess, json

cmd = [
    sys.executable, "scripts/run_experiment.py",
    "--N", "16",
    "--seeder", "dcqo_plus",
    "--device", "cpu",
    "--backend", "cpu",
    "--budget_s", "1.0",
    "--rng_seed", "0",
    "--tag", "cli_demo",
    "--params", json.dumps({
        "steps": 24,
        "trotter_config": {
            "time_grid": "midpoint",           # right_endpoint | midpoint
            "ordering": "forward_reverse",     # fixed | forward_reverse | k_permutation
            "K_perm": 1
        }
    })
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=False)


## 4) DCQO+ trotter knobs quick check (interactive)

This runs a few short DCQO+ experiments with different `trotter_config` settings and compares outputs.

Reminder: DCQO+ upgrades must be **zero-depth** (only discretization/ordering changes; no extra steps/depth).


In [ ]:
import pandas as pd
from labs_hybrid.config import TabuParams

def run_dcqo_plus(trotter_config, tag_suffix):
    run_dir = new_run_dir("results", f"nb_dcqo_plus_{tag_suffix}")
    cfg = RunConfig(
        N=16,
        seeder=SeederConfig(
            name="dcqo_plus",
            shots=256,
            K_out=256,
            K_select=32,
            params={"steps": 24, "trotter_config": trotter_config},
        ),
        backend="cpu",
        device="cpu",
        budget_s=1.0,
        rng_seed=0,
        out_dir=str(run_dir),
        run_tag=f"nb_dcqo_plus_{tag_suffix}",
        save_traces=False,
    )
    write_json(Path(run_dir) / "run_config.json", cfg.to_dict())
    return run_experiment(cfg)

runs = []
runs.append(run_dcqo_plus(
    {"time_grid":"right_endpoint","ordering":"fixed","K_perm":1},
    "baseline"
))
runs.append(run_dcqo_plus(
    {"time_grid":"midpoint","ordering":"fixed","K_perm":1},
    "midpoint"
))
runs.append(run_dcqo_plus(
    {"time_grid":"right_endpoint","ordering":"forward_reverse","K_perm":1},
    "forward_reverse"
))

df = pd.DataFrame([{
    "seeder": r["seeder"],
    "best_seed_energy": r["best_seed_energy"],
    "best_final_energy": r["best_final_energy"],
    "t_compile": r["t_seeder_compile_s"],
    "t_sample": r["t_seeder_sampling_s"],
    "t_total": r["t_seeder_total_s"],
    "trotter_config": r["seeder_info"].get("trotter_config", None),
} for r in runs])

df


## 5) Optional: CPU ↔ GPU energy parity check (only if you have a CUDA GPU)

This directly compares the batched LABS energy function on CPU (NumPy) vs GPU (CuPy).


In [ ]:
import numpy as np
from labs_hybrid.labs_energy_cpu import labs_energy_batch
from labs_hybrid.labs_energy_gpu import labs_energy_batch_gpu

if cp is None:
    print("Skipping parity check (CuPy unavailable).")
else:
    try:
        if int(cp.cuda.runtime.getDeviceCount()) <= 0:
            print("Skipping parity check (no CUDA device).")
        else:
            rng = np.random.default_rng(0)
            K, N = 64, 32
            seeds = rng.choice(np.array([-1, 1], dtype=np.int8), size=(K, N)).astype(np.int8)

            E_cpu = labs_energy_batch(seeds)
            E_gpu = cp.asnumpy(labs_energy_batch_gpu(cp.asarray(seeds, dtype=cp.int8)))

            ok = np.array_equal(E_cpu, E_gpu)
            print("Parity OK?" , ok)
            if not ok:
                # show first mismatch
                idx = np.where(E_cpu != E_gpu)[0][0]
                print("Mismatch at", idx, "CPU", E_cpu[idx], "GPU", E_gpu[idx])
    except Exception as e:
        print("GPU parity check failed:", repr(e))


## 6) Optional: run the full Gate 0→3 benchmark plan

⚠️ This can take time and GPU budget.

By default we keep it **off** in the notebook; flip `RUN_BENCH=True` if you want to run it.


In [ ]:
RUN_BENCH = False

if RUN_BENCH:
    cmd = [sys.executable, "scripts/run_bench_matrix.py", "--tag", "nb_bench"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=False)
else:
    print("RUN_BENCH is False. To run the full plan, set RUN_BENCH=True and re-run this cell.")


## 7) Package a “handoff bundle” zip for AI feedback loop

Each run directory can be zipped into a timestamped bundle using `scripts/package_handoff.py`.

Set `RUN_DIR_TO_PACKAGE` to the run folder you want to share.


In [ ]:
from pathlib import Path

# Example: package the last python-driven run from Section 2
RUN_DIR_TO_PACKAGE = Path(row["out_dir"])

cmd = [sys.executable, "scripts/package_handoff.py", "--run_dir", str(RUN_DIR_TO_PACKAGE)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=False)

print("Look for a .zip inside:", RUN_DIR_TO_PACKAGE)
